# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [1]:
import numpy as np
import pandas as pd

# Load the raw datasets
df = pd.read_csv('data/AviationData.csv', encoding='iso-8859-1', low_memory=False)
states_df = pd.read_csv('data/USState_Codes.csv')

# Simple check to see if data loaded correctly
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [4]:
# Convert event date to datetime format
df['Event.Date'] = pd.to_datetime(df['Event.Date'])

# Filter data to keep records strictly between 1983 and 2023 (the 40-year window)
df = df[(df['Event.Date'].dt.year >= 1983) & (df['Event.Date'].dt.year <= 2023)]

# Force uniform uppercase strings to fix casing typos
df['Make'] = df['Make'].str.upper().str.strip()
df['Model'] = df['Model'].str.upper().str.strip()

# Keep ONLY professional builds (No amateur homebuilt planes)
df = df[df['Amateur.Built'].str.upper().str.strip() == 'NO']

# Keep ONLY actual airplanes (excludes helicopters, gliders, balloons)
df = df[df['Aircraft.Category'].str.upper().str.strip() == 'AIRPLANE']

# Print the remaining rows to verify the filter worked
print(f"Number of remaining airplane records: {len(df)}")

Number of remaining airplane records: 21447


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [5]:
# 1. List the demographic columns related to people on board
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Imputation Assumption: Missing values (NaN) mean zero occurrences for that injury category
for col in injury_cols:
    df[col] = df[col].fillna(0)

# 2. Estimate the total number of passengers on each flight by summing all outcomes
df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# 3. Calculate the likelihood of serious/fatal injury as a fraction of the total population on board
# This gives us a normalized rate between 0.0 and 1.0
df['Casualty.Rate'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']

# Handle division-by-zero errors for records where the total passenger count was zero (e.g., empty parked planes)
df['Casualty.Rate'] = df['Casualty.Rate'].fillna(0)

# Display a preview of the newly constructed metrics
df[['Total.Passengers', 'Casualty.Rate']].head()

,Total.Passengers,Casualty.Rate
4149,588.0,0.0
4150,588.0,0.0
4171,2.0,1.0
4285,5.0,0.2
5957,289.0,0.0


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [6]:
# 1. Standardize text strings by converting to uppercase and removing trailing spaces
df['Aircraft.damage'] = df['Aircraft.damage'].str.upper().str.strip()

# 2. Impute missing damage descriptions with 'UNKNOWN'
df['Aircraft.damage'] = df['Aircraft.damage'].fillna('UNKNOWN')

# 3. Create a derived binary column tracking whether the aircraft was completely destroyed (1) or not (0)
df['Is_Destroyed'] = (df['Aircraft.damage'] == 'DESTROYED').astype(int)

# Preview the results to confirm changes
df[['Aircraft.damage', 'Is_Destroyed']].value_counts(dropna=False)

Aircraft.damage  Is_Destroyed
SUBSTANTIAL      0               16990
DESTROYED        1                2316
UNKNOWN          0                1324
MINOR            0                 817
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [7]:
# 1. Drop any rows where the 'Make' column is entirely missing (NaN)
df = df.dropna(subset=['Make'])

# 2. Identify the frequency of each aircraft manufacturer
make_counts = df['Make'].value_counts()

# 3. Filter the dataframe to keep only Makes with 50 or more records for statistical robustness
robust_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(robust_makes)]

# Preview the top cleaned, high-sample makes remaining in the dataset
df['Make'].value_counts()

Make
CESSNA                            7146
PIPER                             3989
BEECH                             1431
BOEING                            1264
MOONEY                             363
AIRBUS                             243
CIRRUS DESIGN CORP                 220
BELLANCA                           219
AIR TRACTOR INC                    219
MAULE                              215
AIR TRACTOR                        206
AERONCA                            200
CHAMPION                           158
EMBRAER                            153
GRUMMAN                            147
LUSCOMBE                           141
CIRRUS                             137
STINSON                            129
MCDONNELL DOUGLAS                  108
NORTH AMERICAN                     106
DEHAVILLAND                         95
TAYLORCRAFT                         93
AERO COMMANDER                      90
AVIAT AIRCRAFT INC                  76
SOCATA                              75
DIAMOND AIRCRAFT IND

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [8]:
# 1. Drop any rows where the 'Model' column is missing (NaN)
df = df.dropna(subset=['Model'])

# 2. Check if model names are unique across different makes
# We do this by seeing if any Model is associated with more than one Make
model_make_mapping = df.groupby('Model')['Make'].nunique()
overlapping_models = model_make_mapping[model_make_mapping > 1]

print(f"Number of model labels shared by multiple manufacturers: {len(overlapping_models)}")

# 3. Create a derived column that acts as a true unique identifier for each plane type
df['Unique_Aircraft_ID'] = df['Make'] + " " + df['Model']

# Preview the unique identifiers and their active event counts
df['Unique_Aircraft_ID'].value_counts().head(10)

Number of model labels shared by multiple manufacturers: 118


Unique_Aircraft_ID
CESSNA 172     769
BOEING 737     403
CESSNA 152     316
CESSNA 182     304
CESSNA 172S    276
PIPER PA28     273
CESSNA 172N    249
CESSNA 180     213
CESSNA 172M    180
CESSNA 150     179
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [9]:
# 1. Standardize and handle missing values uniformly as 'UNKNOWN' across categorical vectors
categorical_cols = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']

for col in categorical_cols:
    df[col] = df[col].str.upper().str.strip().fillna('UNKNOWN')

# 2. Normalize and clean the 'Number.of.Engines' column
# Imputation Assumption: Missing engine fields default to a baseline of 1 engine, cast safely to integers
df['Number.of.Engines'] = df['Number.of.Engines'].fillna(1).astype(int)

# Preview the unique category mappings for a couple of the cleaned columns to confirm formatting
print(df['Weather.Condition'].value_counts())
print("\n--- Engine Type Cleaned Value Counts ---")
print(df['Engine.Type'].value_counts().head(5))

Weather.Condition
VMC        14295
UNKNOWN     2417
IMC          905
UNK          262
Name: count, dtype: int64

--- Engine Type Cleaned Value Counts ---
Engine.Type
RECIPROCATING    12835
UNKNOWN           3319
TURBO PROP         931
TURBO FAN          701
TURBO JET           71
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [10]:
# 1. Identify and drop columns with an excessive number of missing values (NaNs)
# This includes administrative metadata columns that don't add analytical value for our client
cols_to_drop = ['Air.carrier', 'Schedule', 'FAR.Description', 'Rept.Date']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# 2. Save the fully cleaned, transformed DataFrame to a new CSV file
# Saving without the index to keep the file tidy
df.to_csv('data/Cleaned_Aviation_Data.csv', index=False)

# Confirming everything is saved and showing final dataset dimensions
print(f"Data cleaning complete! Final dataset shape: {df.shape}")
print("Successfully exported to 'data/Cleaned_Aviation_Data.csv'.")

Data cleaning complete! Final dataset shape: (17879, 32)
Successfully exported to 'data/Cleaned_Aviation_Data.csv'.


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [11]:
# Save the fully cleaned and engineered DataFrame to your project directory
df.to_csv('data/Cleaned_Aviation_Data.csv', index=False)

# Quick check to ensure the output matches our expectations
print("All cleaning steps successfully processed and verified!")
print(f"Cleaned dataset saved with {df.shape[0]} records and {df.shape[1]} columns.")

All cleaning steps successfully processed and verified!
Cleaned dataset saved with 17879 records and 32 columns.
